# Classifying Tomato Diseases with YOLO Image Classification

## Introduction: the problem

Plant diseases cause major losses in tomato production, and the right control measure depends
on the **correct diagnosis**. A fungal disease needs a different response from a viral or a
vascular one. Diagnosis by eye needs experienced people and is slow over large areas. Image-based
deep-learning models could give growers a fast first diagnosis from a phone photo.

This dataset covers three very different diseases:

| Disease | Cause | Typical symptoms |
|---|---|---|
| **Gray mold** | Fungus *Botrytis cinerea*, favoured by high humidity | Water-soaked lesions that become covered by a gray, fuzzy layer of spores, on leaves, stems and fruit |
| **Viral** diseases | Viruses (e.g. tomato yellow leaf curl, tomato mosaic) spread by insects or contact | Mosaic/mottled or yellow leaves, leaf curling and distortion, stunted shoot tips |
| **Wilt** | Soil-borne vascular pathogens (e.g. *Fusarium*, *Verticillium*, *Ralstonia*) that block water transport | Drooping and yellowing leaves, browning of stems, collapse of the plant |

### What will the model do?
This notebook trains an **image classifier**. The model looks at the *whole* photo and outputs a
single label (`GrayMold`, `Viral` or `Wilt`) with a confidence score. Unlike object detection
(see the three companion detection notebooks), it does **not** say *where* the symptoms are.
In exchange it needs **no box annotations**: the folder each image is in *is* its label.

```
Tomato Disease Dataset/images/
 ├── GrayMold/   (82 images)
 ├── Viral/      (417 images)
 └── Wilt/       (527 images)
```

> **Class imbalance:** there are ~6× more Wilt than Gray Mold images. A model that ignores
> Gray Mold entirely can still reach a high *overall* accuracy, so we also report
> **per-class** results and a confusion matrix.

### Learning objectives
By the end of this notebook you should be able to:
1. Prepare an image dataset for classification (folder structure, train/val/test split).
2. Fine-tune a pretrained YOLO classifier (**transfer learning**).
3. Evaluate a classifier with accuracy, per-class recall/precision and a confusion matrix.
4. Run a small experiment (change one setting, compare results) and interpret it.

### Roadmap
| Step | What happens |
|---|---|
| 1 | Set up Colab (GPU, libraries) |
| 2 | Download the dataset |
| 3 | Build the train / val / test folders |
| 4 | Train the classifier (**settings you can change are here**) |
| 5 | Inspect the training curves |
| 6 | Evaluate on validation and **test** sets |
| 7 | Look at individual predictions |
| 8 | Per-class results and confusion matrix |
| 9 | Log the experiment |
| Final | Instructions for your report |

> Throughout the notebook, notes marked **For your report** tell you what to write down.

## Key concepts

| Term | Meaning |
|---|---|
| **Image classification** | Assign one label to a whole image |
| **Top-1 accuracy** | Fraction of images whose most likely predicted class is correct |
| **Top-5 accuracy** | Fraction where the true class is among the 5 most likely. With only 3 classes this is always 1.0, so ignore it here |
| **Recall** (per class) | Of the images that truly have disease X, the fraction predicted as X: *"how many did we miss?"* |
| **Precision** (per class) | Of the images predicted as X, the fraction that really are X: *"how many false alarms?"* |
| **Confusion matrix** | Table of true class (rows) × predicted class (columns). The diagonal holds the correct predictions |
| **Transfer learning** | Start from a network pretrained on ImageNet (1000 everyday object classes) and fine-tune it on our images |
| **Epoch** | One full pass through the training images |
| **Train / validation / test** | *Train*: the model learns from these. *Validation*: used during training to pick the best epoch. *Test*: never seen during training, so it gives an unbiased final score |

# 1. Setup

### 1.1 Turn on a GPU
In Colab, open `Runtime > Change runtime type`, choose **T4 GPU** (or any GPU) and click `Save`.
Training on a CPU works, but it is roughly 10-50× slower.

### 1.2 Install and import
The next cells check the GPU, install the **Ultralytics** library (which provides YOLO) and
import the Python packages used in the notebook. If `nvidia-smi` prints an error, you are not on
a GPU runtime.

In [ ]:
!nvidia-smi

**Install Ultralytics**, the library that provides YOLO. `-U` upgrades it to the latest version,
so the newest YOLO models (YOLO26) are available.

In [ ]:
!pip install -q -U ultralytics

**Import the libraries** used in the notebook and choose the device: GPU 0 if one is available,
otherwise the CPU. The cell prints the device that will be used for training.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os, glob, random, shutil
from pathlib import Path
import torch

DEVICE = 0 if torch.cuda.is_available() else "cpu"   # GPU 0 if available, else CPU
print("Ultralytics YOLO loaded. Training device:", DEVICE)

# 2. Get the data

The photos come from the **Tomato Disease Dataset** on Mendeley Data:
https://data.mendeley.com/datasets/c2x8rynybg/1 (open this page to read the description of the
dataset).

You do not need to download anything yourself: the download link
(`https://data.mendeley.com/public-api/zip/c2x8rynybg/download/1`) is already in the cell below.
The cell downloads the archive (about 3 GB, a few minutes) into Colab and unzips it.

It then finds the dataset root (the folder containing `images/`) and prints its location.
Inside, `images/` has one sub-folder per disease (`GrayMold`, `Viral`, `Wilt`); the folder an image
is in is its label.

In [ ]:
DATA_URL = "https://data.mendeley.com/public-api/zip/c2x8rynybg/download/1"   # Google Drive share link OR a direct URL
ARCHIVE  = "dataset.zip"                      # local filename to save the download as

!pip install -q gdown
os.makedirs("/content/data", exist_ok=True)

# Download: gdown parses Google Drive share links (--fuzzy); wget handles any direct URL
if "drive.google.com" in DATA_URL or "docs.google.com" in DATA_URL:
    !gdown --fuzzy "{DATA_URL}" -O {ARCHIVE}
else:
    !wget -q --show-progress -O {ARCHIVE} "{DATA_URL}"

# Unzip if it is a zip archive
if ARCHIVE.lower().endswith(".zip"):
    !unzip -qo {ARCHIVE} -d /content/data
else:
    !cp {ARCHIVE} /content/data/ 2>/dev/null

# Auto-detect the dataset root = folder that contains images/<disease>/ sub-folders
# (skip the __MACOSX folder that zips made on a Mac often contain)
cands = sorted(os.path.dirname(p) for p in glob.glob("/content/data/**/images", recursive=True)
               if "__MACOSX" not in p)
assert cands, "No 'images' folder found - check DATA_URL and the unzip output above."
DATA_ROOT = cands[0]
print("DATA_ROOT =", DATA_ROOT)
!ls "{DATA_ROOT}/images"

# 3. Build the classification dataset

YOLO classification expects one sub-folder per class inside each split:

```
tomato_cls/
 ├── train/<class>/*.jpg
 ├── val/<class>/*.jpg
 └── test/<class>/*.jpg
```

We split **each disease separately** 70 / 20 / 10 (a *stratified* split), so all three
diseases appear in every split in the same proportions. A fixed random seed (42) makes the
split identical for everyone.

We also **bake the EXIF orientation into the pixels**. Many phone photos are stored sideways with
a "rotate" flag; we rotate them permanently so every program sees the plant upright.

> **For your report (Methodology):** record the number of images per class in each split
> (printed below). How imbalanced is the dataset?
>
> **Note:** The split is random *per image*. Photos taken on the same day of the same plant can end up
> in different splits, which can make the test accuracy look better than it really is. Keep this
> in mind for the discussion.

In [ ]:
# import PIL's Image as PILImage so we don't shadow IPython.display.Image (used later)
from PIL import Image as PILImage, ImageOps
PILImage.MAX_IMAGE_PIXELS = None

SRC = os.path.join(DATA_ROOT, "images")      # DATA_ROOT set in the download cell
OUT = "/content/tomato_cls"
CLASSES = sorted(d for d in os.listdir(SRC) if os.path.isdir(os.path.join(SRC, d)))
SPLITS  = {"train": 0.70, "val": 0.20, "test": 0.10}
IMG_EXTS = (".jpg", ".jpeg", ".png")
random.seed(42)
print("Classes:", CLASSES)

shutil.rmtree(OUT, ignore_errors=True)   # start clean so re-runs never mix old and new splits
counts = {s: {} for s in SPLITS}
for c in CLASSES:
    # sorted -> same file order (and therefore the same random split) on every machine
    imgs = sorted(p for p in glob.glob(os.path.join(SRC, c, "*"))
                  if p.lower().endswith(IMG_EXTS) and not Path(p).name.startswith("._"))
    random.shuffle(imgs)
    n = len(imgs); n_tr = int(n * SPLITS["train"]); n_va = int(n * SPLITS["val"])
    for i, p in enumerate(imgs):
        split = "train" if i < n_tr else ("val" if i < n_tr + n_va else "test")
        dst = os.path.join(OUT, split, c)
        os.makedirs(dst, exist_ok=True)
        # bake EXIF rotation into pixels so every loader/viewer sees the same orientation
        img = ImageOps.exif_transpose(PILImage.open(p)).convert("RGB")
        img.save(os.path.join(dst, Path(p).stem + ".jpg"), quality=95)
        counts[split][c] = counts[split].get(c, 0) + 1

import pandas as pd
split_table = pd.DataFrame(counts).reindex(CLASSES).fillna(0).astype(int)
split_table.loc["total"] = split_table.sum()
split_table

### Preview one image per class
**Always look at your data.** Can *you* tell the three diseases apart? Which visual cues would
a model have to learn? Re-run the cell to see other examples.

In [ ]:
import cv2
import matplotlib.pyplot as plt
%matplotlib inline

fig, axes = plt.subplots(1, len(CLASSES), figsize=(14, 5))
for ax, c in zip(axes, CLASSES):
    p = random.choice(glob.glob(os.path.join(OUT, "train", c, "*")))
    ax.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
    ax.set_title(c); ax.axis("off")
plt.tight_layout(); plt.show()

# 4. Train the classifier

### 4.1 Experiment settings: **this is the cell you change**

Training starts from a YOLO classifier **pretrained on ImageNet** (transfer learning). The
network already recognises general visual features (edges, textures, colours) and only needs
to learn which ones separate the three diseases. Classification uses smaller images (224 px)
than detection, so it trains quickly.

| Setting | What it does | Suggested values to try |
|---|---|---|
| `MODEL_NAME` | Model size: **n**ano < **s**mall < **m**edium < **l**arge | `yolo26n-cls.pt` (default), `yolo26s-cls.pt`, `yolo26m-cls.pt` |
| `EPOCHS` | Maximum number of passes over the training data | 30, 50 (default), 100 |
| `PATIENCE` | Early stopping: stop if validation accuracy has not improved for this many epochs | 15 (default), 30 |
| `IMGSZ` | Images are resized to this many pixels. Larger keeps fine symptom detail but is slower | 224 (default), 320, 448 |
| `BATCH` | Images per training step. Lower it if you get a *CUDA out of memory* error | 32 (default), 16 |

> **Do a proper experiment:** change **one** setting at a time and compare against your
> baseline. Each run gets its own name (`RUN_NAME`) and folder, so results never overwrite each
> other.

In [ ]:
# ======================= EXPERIMENT SETTINGS (edit me) =======================
MODEL_NAME = "yolo26n-cls.pt"   # yolo26n-cls.pt / yolo26s-cls.pt / yolo26m-cls.pt
EPOCHS     = 50
PATIENCE   = 15
IMGSZ      = 224
BATCH      = 32
# ==============================================================================
RUN_NAME   = f"cls_{Path(MODEL_NAME).stem}_e{EPOCHS}_img{IMGSZ}_b{BATCH}"
PROJECT    = "/content/runs/classify_disease"
print("Run name:", RUN_NAME)

### 4.2 Train
**Check the installation.** `yolo checks` prints the software and hardware versions (useful for the *Methodology*).

In [ ]:
!yolo checks

**Train the model.** The cell loads the pretrained weights given in `MODEL_NAME`. If that model is not available in the installed
Ultralytics version, it falls back to the equivalent YOLO11 model and says so. It then trains with the
settings from the experiment-settings cell. During
training, each epoch prints the loss and the validation `top1_acc`. The best epoch on the
validation set is saved as **`best.pt`**, which we use from here on.

In [ ]:
try:
    model = YOLO(MODEL_NAME)
except Exception as e:                       # YOLO26 needs a recent ultralytics version
    fallback = MODEL_NAME.replace("yolo26", "yolo11")
    print(f"'{MODEL_NAME}' unavailable ({e}); falling back to {fallback}")
    MODEL_NAME = fallback
    model = YOLO(MODEL_NAME)

train_results = model.train(
    data=OUT,            # folder with train/ val/ test/ sub-folders
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=DEVICE,       # GPU 0 if available, else "cpu"
    seed=42,
    project=PROJECT,
    name=RUN_NAME,       # each experiment gets its own folder
)

RUN_DIR = Path(model.trainer.save_dir)        # actual folder used (e.g. ..._2 if the name existed)
BEST_PT = RUN_DIR / "weights" / "best.pt"
print("Run folder:", RUN_DIR)

# 5. Training curves

`results.png` shows, per epoch, the **training loss** and **validation loss** (should go down)
and the validation **top-1 accuracy** (should go up and level off). If the training loss keeps
falling while the validation loss rises, the model is **over-fitting**.

> **For your report (Results):** include this figure for your best run. At which epoch did
> it reach its best accuracy, and did training stop early?

In [ ]:
results_img = RUN_DIR / "results.png"
if results_img.exists():
    display(Image(filename=str(results_img), width=1000))
else:
    print("results.png not found in", RUN_DIR)

hist = pd.read_csv(RUN_DIR / "results.csv")
hist.columns = hist.columns.str.strip()
best_row = hist.loc[hist["metrics/accuracy_top1"].idxmax()]
print(f"Epochs run: {len(hist)} | best val top-1 = {best_row['metrics/accuracy_top1']:.3f} at epoch {int(best_row['epoch'])}")

# 6. Evaluate on the validation and **test** sets

We reload `best.pt` and compute top-1 accuracy on the **validation** split (used to pick the
best epoch, so slightly optimistic) and on the **test** split (never seen during training, so
these are your final numbers).

In [ ]:
model = YOLO(str(BEST_PT))
val_metrics  = model.val(data=OUT, split="val",  imgsz=IMGSZ, device=DEVICE,
                         project=PROJECT, name=f"{RUN_NAME}_val")
test_metrics = model.val(data=OUT, split="test", imgsz=IMGSZ, device=DEVICE,
                         project=PROJECT, name=f"{RUN_NAME}_test")
print(f"VALIDATION top-1 accuracy = {val_metrics.top1:.3f}")
print(f"TEST       top-1 accuracy = {test_metrics.top1:.3f}")

# 7. Look at individual predictions

The model outputs a probability for each class. The predicted label is the most likely one. The
cell shows a few random test images with the predicted label, its confidence and the true
label (red title = wrong). Re-run it to see other images.

> **For your report:** keep at least one **misclassified** example. What might have misled
> the model (lighting, background, early symptoms, several symptoms in one photo)?

In [ ]:
test_imgs = sorted(glob.glob(os.path.join(OUT, "test", "*", "*.jpg")))
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, sample in zip(axes, random.sample(test_imgs, 4)):
    r = model.predict(sample, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
    pred, conf, true = r.names[r.probs.top1], float(r.probs.top1conf), Path(sample).parent.name
    ax.imshow(cv2.cvtColor(cv2.imread(sample), cv2.COLOR_BGR2RGB)); ax.axis("off")
    ax.set_title(f"pred: {pred} ({conf:.2f})\ntrue: {true}",
                 color="black" if pred == true else "red")
plt.tight_layout(); plt.show()

# 8. Per-class results and confusion matrix (test set)

With imbalanced classes, overall accuracy can hide a weak class. We predict every test image
and compute, per disease, **recall** (how many of that disease were found) and **precision**
(how many predictions of that disease were right), plus the **confusion matrix**.

**Reading the confusion matrix:** each row is the true disease and each column is the predicted
one. The diagonal holds correct predictions; anything off the diagonal is a confusion between two
diseases.

> **For your report (Results):** the per-class table is your main results table (Table 2)
> and the confusion matrix is a key figure. Which diseases get confused with each other?

In [ ]:
import numpy as np

y_true, y_pred, files = [], [], []
for p in test_imgs:
    r = model.predict(p, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
    y_true.append(Path(p).parent.name)
    y_pred.append(r.names[r.probs.top1])
    files.append(Path(p).name)
y_true, y_pred = np.array(y_true), np.array(y_pred)

labels = CLASSES
cm = pd.crosstab(pd.Series(y_true, name="true"),
                 pd.Series(y_pred, name="pred")).reindex(index=labels, columns=labels, fill_value=0)

per_class = pd.DataFrame({
    "test_images": [(y_true == c).sum() for c in labels],
    "correct":     [((y_true == c) & (y_pred == c)).sum() for c in labels],
    "recall":      [((y_true == c) & (y_pred == c)).sum() / max((y_true == c).sum(), 1) for c in labels],
    "precision":   [((y_true == c) & (y_pred == c)).sum() / max((y_pred == c).sum(), 1) for c in labels],
}, index=labels).round(3)
per_class.to_csv(RUN_DIR / "per_class_test_metrics.csv")

acc = (y_true == y_pred).mean()
print(f"Overall test accuracy: {acc:.3f}   |   mean per-class recall (balanced accuracy): {per_class['recall'].mean():.3f}")
print("Misclassified test images:", [f for f, t, q in zip(files, y_true, y_pred) if t != q])
per_class

**Confusion matrix figure.** The same confusion matrix as a colour-coded heatmap: rows are the
true disease, columns the predicted one, and the numbers are image counts. The figure is also
saved as `confusion_matrix_test.png` in the run folder.

In [ ]:
import seaborn as sns
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion matrix (test set)"); plt.ylabel("true"); plt.xlabel("predicted")
plt.tight_layout(); plt.savefig(RUN_DIR / "confusion_matrix_test.png", dpi=150); plt.show()

# 9. Log this experiment

Every time you run the notebook with different settings, the cell below adds **one row** to
`experiment_log.csv`. After a few runs, this table is the basis for your *experiment* table in
the report.

> **Colab deletes `/content` when the session ends.** Download the log, figures
> (`results.png`, `confusion_matrix_test.png`) and `best.pt` (Files panel on the left,
> right-click, *Download*), or copy them to Google Drive.

In [ ]:
log_path = Path(PROJECT) / "experiment_log.csv"
entry = pd.DataFrame([{
    "run": RUN_DIR.name, "model": MODEL_NAME, "epochs_max": EPOCHS, "epochs_run": len(hist),
    "best_epoch": int(best_row["epoch"]), "imgsz": IMGSZ, "batch": BATCH, "patience": PATIENCE,
    "val_top1": round(val_metrics.top1, 3), "test_top1": round(test_metrics.top1, 3),
    "test_balanced_acc": round(per_class["recall"].mean(), 3),
    **{f"recall_{c}": per_class.loc[c, "recall"] for c in labels},
}])
log = pd.concat([pd.read_csv(log_path), entry]) if log_path.exists() else entry
log = log.drop_duplicates(subset="run", keep="last")
log.to_csv(log_path, index=False)
print("Saved", log_path)
log

# 10. Summary

In this notebook you:
- built a stratified, reproducible train/val/test split from the dataset's disease folders;
- fine-tuned a pretrained YOLO classifier to recognise Gray Mold, Viral and Wilt;
- evaluated it on validation and **test** data with overall accuracy, per-class
  recall/precision and a confusion matrix;
- logged the experiment.

**Ideas for improving results:** train longer, try a larger model (`yolo26s-cls.pt`) or a larger
`IMGSZ` to keep fine symptom detail. To handle the imbalance (advanced), add more Gray Mold
images or oversample them in the training folder.

# Assignment: write your report

You now have everything you need. Your task is to **get the best results you can** by making
small changes to the settings in **Step 4.1** (e.g. `EPOCHS`, `MODEL_NAME`, `IMGSZ`), and then to
describe the study in a **written report of about 2 pages**. You will also present your findings
to the class in a short **PowerPoint presentation** built from the same material.

### Before you write
1. Run the notebook once with the default settings. This is your **baseline**.
2. Run **at least 2 more experiments**, changing **one setting at a time** so you can tell what
   caused any difference. Each run is added to `experiment_log.csv` (Step 9).
3. **Download** your log, figures and best weights before the Colab session ends (`/content`
   is wiped when the runtime disconnects).

### Report structure

**1. Introduction (~¼ page)**
- The plant-pathology problem: the disease, its cause and symptoms, and why it matters.
- Why automated, image-based diagnosis is useful (scale, speed, early detection).
- What classification of tomato images into **Gray Mold, Viral or Wilt** means, in one or two sentences.
- **The objective of your study**, stated clearly in the last sentence of the introduction.

**2. Methodology (~½ page)**: give enough detail that someone else could repeat your work.
- **Dataset:** source, number of images, classes, number of instances per class, imbalance.
- **Pre-processing:** annotation/format conversion, EXIF orientation correction,
  train/validation/test split (70/20/10, random seed 42) and what each split is used for.
- **Model:** YOLO architecture and size, pretrained weights and **transfer learning**.
- **Training settings:** epochs, early-stopping patience, image size, batch size, hardware (GPU).
- **Evaluation metrics:** top-1 accuracy, per-class recall (sensitivity), the confusion matrix, and why overall accuracy can be misleading with imbalanced classes.

**3. Results (~½-¾ page)**: report *what you observed*; leave the interpretation for the
discussion. You can use tables and figures, or describe the results in short sentences. Number every
table and figure, give each a caption, and refer to it in the text.

**4. Discussion and conclusion (~½ page)**: interpret your results. You do not have to answer all of these questions; use them as a guide:

- Which disease is recognised best and which worst? Which diseases get **confused** with each
  other, and is that biologically plausible (similar symptoms)?
- How does the **class imbalance** (Wilt 527 vs Gray Mold 82 images) affect the results? Is
  overall accuracy a fair summary, or does per-class recall tell a different story?
- What did changing the epochs / model size / image size do? Any signs of **over-fitting**?
- **Limitations:** small and imbalanced dataset, a single random split. Photos taken on the
  same date often show the *same plant*, so near-duplicates can land in both train and test and
  inflate accuracy. Every image contains exactly one disease, but real plants can have several.
- **Classification vs detection:** what does a classifier *not* tell a grower that a detector
  (the other notebooks) does? When is each approach more useful?
- Would this model be usable in practice? What would improve it?

End with a short **conclusion** (2-3 sentences) that directly answers the objective stated in
your introduction, and suggest one or two concrete next steps.

**5. References**: cite the dataset, Ultralytics YOLO (https://docs.ultralytics.com) and any
plant-pathology sources you used for the introduction.

If anything in the notebook does not run, contact the instructors. We are happy to help.